<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/rapidsandfilter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Rapid Sand Filtration calculation***


In [ ]:
import math as mh
import pandas as pd
def rapidsandfilter(lanes,Partable2,dailyflow,Treatment):
    # Rapid Sand filtration design
    # This code calculates the dimensions of rapid sand filtration stage in a water treatment plant.

    # Input parameters
    print("")
    print("\nRAPID SAND FILTRATION PROCESS")

    # Initialize all variables that might be used in multiple branches
    surfaceLoadingRatebackwash = None
    filterAreaperunit = None
    filterArea = None
    units = None

    flowRate = dailyflow/24  # m^3/h
    print(f"The nominal flow rate  is {flowRate:.3f} m3/h")
    watervisc=0.0010005 # dynamic viscosity of water in N*s/m^2
    rho = 1000 # density of water kg/m3
    waterkinvisc=0.0010005/1000 # kinematic viscosity of water in m^2/s
    g= 9.81 # Acceleration due to gravity (m/s^2)
    sandpor=0.4 # porosity of sand filters
    sanddens=2650 # density of sand kg/m3
    anthrpor=0.52 # porosity of anthracite filters
    anthrdens=1450 # density of anthracite kg/m3


    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "single media filtration").any():

        print("The type of media selected for single media filtration is sand")

        while True:
            media_depth = float(input("Enter the media depth in meters (0.6m min - 0.8m max - default 0.6m): ")or 0.6)
            try:
               if 0.6 <= media_depth <= 0.8:
                  break
               else:
                  print("Invalid input. Please enter a value between 0.6 and 0.8.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        bed_depth = mh.ceil(media_depth*1.2)

        while True:
            grain_size = float(input("Enter the media grain size millimeters (0.4mm min - 0.6mm max - default 0.6mm): ")or 0.6)
            try:
               if 0.4 <= grain_size <= 0.6:
                  break
               else:
                  print("Invalid input. Please enter a value between 0.4 and 0.6.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")

        min_depth = 0.6  # Minimum depth in meters
        max_depth = 0.9  # Maximum depth in meters
        min_rate = 8.0   # Minimum filtration rate in m/h
        max_rate = 15.0  # Maximum filtration rate in m/h

        while True:
            try:
               surfaceLoadingRate = int(input("Required hydraulic loading rate (8m/h min - 12m/h max - default 10m): ")or 10)
               if 8 <= surfaceLoadingRate <= 12:
                   while True:
                       units_str = input("Number of different rapid sand filtration units per lane (minimum 1 unit per lane):")
                       try:
                          units = int(units_str)
                          if units >= 1:
                              break
                          else:
                              print("Invalid input. Please enter a number greater than or equal to 1.")
                       except ValueError:
                           print("Invalid input. Please enter a valid number.")
                   filterArea = flowRate / surfaceLoadingRate  # m2
                   filterAreaperunit = filterArea / (lanes*units)
                   filterAreaperunit = mh.ceil(filterAreaperunit)
                   print(f"Initial calculated surface area per unit: {filterAreaperunit} m²")
                   filterArea=filterAreaperunit*lanes*units
                   print(f"Total required rapid sand filtration surface area : {filterArea} m²")
                   num_operational_tanks_backwash = lanes*units - 1
                   print(f"Number of operational units during backwash: {num_operational_tanks_backwash}")
                   flowRatebackwashunit = flowRate / num_operational_tanks_backwash
                   surfaceLoadingRatebackwash = flowRatebackwashunit/filterAreaperunit
                   if surfaceLoadingRatebackwash >= max_rate:
                      print(f"Warning: Hydraulic loading rate per unit during backwash ({surfaceLoadingRatebackwash} m/h) exceeds the maximum limit of {max_rate} m/h.")
                      print(f"The number of tanks per lane should increase from {units}  per lane to {units+1}  to manage backwash flow.")
                      units = units + 1
                      filterArea = flowRate / surfaceLoadingRate  # m2
                      filterAreaperunit = filterArea / (lanes*units)
                      filterAreaperunit = mh.ceil(filterAreaperunit)
                      print(f"The re-calculated surface area per unit: {filterAreaperunit} m²")
                      filterArea=filterAreaperunit*lanes*units
                      print(f"The new total required rapid sand filtration surface area is : {filterArea} m²")
                   else:
                      filterArea=filterArea
                      filterAreaperunit=filterAreaperunit
                   break
               else:
                   print("Invalid input. Please enter a value between 8 and 15.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        # Hydraulic losses
        grain_size_m=grain_size/1000 #mm to m
        vel_unit=surfaceLoadingRatebackwash/3600 # superficial velocity in m/s
        term1=waterkinvisc/g
        term2=(1 - sandpor)**2 / sandpor**3
        term3=vel_unit/grain_size_m**2
        rapidsandhead=mh.ceil(180*term1*term2*term3*bed_depth)
        print(f"The hydraulic losses in the rapid sand filtration units is {rapidsandhead} m")

        # Backwash calculation
        print('Assumption: Backwash hydraulic loading rate equal is 50m/h, backwash duration is 10min. Always one unit backwash once per week')
        Qbackwash=filterAreaperunit*50 # 50m3/h/m2 backwash flow rate standard
        t=1/6 #10min backwash time
        Vbackwash=Qbackwash * t
        Vbackwash=mh.ceil(1.1*Vbackwash*lanes*units) # 10% increase in backwash volume
        print(f"The minimum backwash volume required for the backwash of all the rapid sand filtration units is {Vbackwash}m3 ")
        Abackwash=Vbackwash/3
        print(f"The required backwash area is {Abackwash} m2")
        term1=waterkinvisc**0.8/g
        term2=(1 - sandpor)**1.8 / sandpor**3
        term3=vel_unit*1.2/grain_size_m**1.8
        rapidsandbackhead=mh.ceil(1.1*(bed_depth+130*term1*term2*term3*bed_depth))
        print(f"The hydraulic losses during the backwash is {rapidsandbackhead} m")
        # Power and energy calculations
        Etot=0
        Pback=(1000* 9.81*Qbackwash*rapidsandbackhead/3600)/1000 #Power of the backwash pump
        Pback=round(Pback/0.7) # 70% efficiency
        print(f"The power consumption of the backwash pumps is {Pback} kW")
        Eback=Pback*t*lanes*units/7 # Daily 7days projected energy consumption (one day backwash and  days off)
        Eback=mh.ceil(Eback)
        print(f"The energy consumption of the backwash pumps is {Eback} kWh")
        # Total area of the rapid sand filtration process
        rapidfilterprocessArea = mh.ceil(1.2*(filterArea + Abackwash))
        print(f"The total area of the rapid sand filtration process is {rapidfilterprocessArea} m2")

    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "dual media filtration").any():
        print("The type of media for dual media filtration is sand and anthracite")
        while True:
            media_depth1 = float(input("Enter the sand media depth in meters (0.3m min - 0.5m max - default 0.5m): ")or 0.5)
            try:
               if 0.3 <= media_depth1 <= 0.5:
                  break
               else:
                print("Invalid input. Please enter a value between 0.3 and 0.5.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        while True:
            grain_size1 = float(input("Enter the sand media grain size millimeters (0.4mm min - 0.6mm max - default 0.6mm): ")or 0.6)
            try:
               if 0.4 <= grain_size1 <= 0.6:
                  break
               else:
                  print("Invalid input. Please enter a value between 0.4 and 0.6.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        while True:
            media_depth2 = float(input("Enter the anthracite media depth in meters (0.5m min - 0.7m max - default 0.7m): ")or 0.7)
            try:
               if 0.5 <= media_depth2 <= 0.7:
                  break
               else:
                  print("Invalid input. Please enter a value between 0.5 and 0.7.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        while True:
            grain_size2 = float(input("Enter the sand media grain size millimeters (0.6mm min - 1.5mm max - default 1mm): ")or 1)
            try:
               if 0.6 <= grain_size2 <= 1.5:
                  break
               else:
                  print("Invalid input. Please enter a value between 0.6 and 1.5.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        bed_depth = (media_depth1+media_depth2*1.2)
        min_depth = 0.9  # Minimum depth in meters
        max_depth = 1.2  # Maximum depth in meters
        min_rate = 10.0   # Minimum filtration rate in m/h
        max_rate = 20.0  # Maximum filtration rate in m/h
        while True:
            surfaceLoadingRate = int(input("Required hydraulic loading rate (10m/h min - 18m/h max - default 15m): ")or 15)
            try:
               if 10 <= surfaceLoadingRate <= 18:
                   while True:
                       units_str = input("Number of different rapid sand filtration units per lane (minimum 1 unit per lane):")
                       try:
                          units = int(units_str)
                          if units >= 1:
                              break
                          else:
                              print("Invalid input. Please enter a number greater than or equal to 1.")
                       except ValueError:
                           print("Invalid input. Please enter a valid number.")
                   filterArea = flowRate / surfaceLoadingRate  # m2
                   filterAreaperunit = filterArea / (lanes*units)
                   filterAreaperunit = mh.ceil(filterAreaperunit)
                   print(f"Initial calculated surface area per unit: {filterAreaperunit} m²")
                   filterArea=filterAreaperunit*lanes*units
                   print(f"Total required rapid sand filtration surface area : {filterArea} m²")
                   num_operational_tanks_backwash = lanes*units - 1
                   print(f"Number of operational units during backwash: {num_operational_tanks_backwash}")
                   flowRatebackwashunit = flowRate / num_operational_tanks_backwash
                   surfaceLoadingRatebackwash = flowRatebackwashunit/filterAreaperunit
                   if surfaceLoadingRatebackwash >= max_rate:
                      print(f"Warning: Hydraulic loading rate per unit during backwash ({surfaceLoadingRatebackwash} m/h) exceeds the maximum limit of {max_rate} m/h.")
                      print(f"The number of tanks per lane should increase from {units}  per lane to {units+1}  to manage backwash flow.")
                      units = units + 1
                      filterArea = flowRate / surfaceLoadingRate  # m2
                      filterAreaperunit = filterArea / (lanes*units)
                      filterAreaperunit = mh.ceil(filterAreaperunit)
                      print(f"The re-calculated surface area per unit: {filterAreaperunit} m²")
                      filterArea=filterAreaperunit*lanes*units
                      print(f"The new total required rapid sand filtration surface area is : {filterArea} m²")
                   else:
                      filterArea=filterArea
                      filterAreaperunit=filterAreaperunit
                   break
               else:
                   print("Invalid input. Please enter a value between 10 and 20.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")

        # Hydraulic losses
        grain_size_m1=grain_size1/1000 #mm to m
        grain_size_m2=grain_size2/1000 #mm to m
        vel_unit=surfaceLoadingRatebackwash/3600 # superficial velocity in m/s
        term1=waterkinvisc/g
        term22=(1 - anthrpor)**2 / anthrpor**3
        term32=vel_unit/grain_size_m2**2
        rapidsandhead2=180*term1*term22*term32*media_depth2
        term21=(1 - sandpor)**2 / sandpor**3
        term31=vel_unit/grain_size_m1**2
        rapidsandhead1=180*term1*term21*term31*media_depth1
        rapidsandhead=mh.ceil(rapidsandhead1+rapidsandhead2)
        print(f"The hydraulic losses in the rapid sand filtration units is {rapidsandhead} m")

        # Backwash calculation
        print('Assumption: Backwash hydraulic loading rate equal is 50m/h, backwash duration is 10min. Always one unit backwash once per week')
        Qbackwash=filterAreaperunit*50 # 50m3/h/m2 backwash flow rate standard
        t=1/6 #10min backwash time
        Vbackwash=Qbackwash * t
        Vbackwash=mh.ceil(1.1*Vbackwash*lanes*units) # 10% increase in backwash volume
        print(f"The minimum backwash volume required for the backwash of all the rapid sand filtration units is {Vbackwash}m3 ")
        Abackwash=Vbackwash/2
        print(f"The required backwash area is {Abackwash} m2")
        term1=waterkinvisc**0.8/g
        term21=(1 - sandpor)**1.8 / sandpor**3
        term31=vel_unit*1.2/grain_size_m1**1.8
        rapidsandbackhead1=mh.ceil(1.1*(media_depth1+130*term1*term21*term31*media_depth1))
        term22=(1 - anthrpor)**1.8 / anthrpor**3
        term32=vel_unit*1.2/grain_size_m2**1.8
        rapidsandbackhead2=mh.ceil(1.1*(media_depth2+130*term1*term22*term32*media_depth2))
        rapidsandbackhead=rapidsandbackhead1+rapidsandbackhead2
        print(f"The hydraulic losses during the backwash is {rapidsandbackhead} m")
        # Power and energy calculations
        Etot=0
        Pback=(1000* 9.81*Qbackwash*rapidsandbackhead/3600)/1000 #Power of the backwash pump
        Pback=round(Pback/0.7) # 70% efficiency
        print(f"The power consumption of the backwash pumps is {Pback} kW")
        Eback=Pback*t*lanes*units/7 # Daily 7days projected energy consumption (one day backwash and  days off)
        Eback=mh.ceil(Eback)
        print(f"The energy consumption of the backwash pumps is {Eback} kWh")
        # Total area of the rapid sand filtration process
        rapidfilterprocessArea = mh.ceil(1.2*(filterArea + Abackwash))
        print(f"The total area of the rapid sand filtration process is {rapidfilterprocessArea} m2")

    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "pressurized filtration vessel").any():

        while True:
            units_str = input("Number of different units per lane (minimum 1+1 back up):")
            try:
               units = int(units_str)
               if units >= 2:
                    break
               else:
                   print("Invalid input. Please enter a number greater than or equal to 1.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")

        V=flowRate/(lanes*(units-1))
        data = {
         'Height (m)': [1.22, 1.22, 1.22, 1.5, 1.5, 1.5, 1.5, 1.8, 2.0, 2.0, 2.0, 2.5],
         'Diameter (mm)': [500, 600, 800, 900, 1000, 1200, 1400, 1600, 1800, 2500, 2800, 3200],
         'Filtration flow (m3/h)': [(0.001, 3), (3, 5), (5, 8),(8, 10), (10, 12),(12, 17), (17, 24),(24, 30), (30, 47),(47, 68), (68, 92), (92, 120)]}
        # Create DataFrame
        df = pd.DataFrame(data)

        # Print the DataFrame for debugging
        print("Pressurized Filtration Vessels Technical parameters:")
        print(df)

        # Find the appropriate line based on the given flow rate
        selected_filter = df[(df['Filtration flow (m3/h)'].apply(lambda x: x[0]) < V) &
                              (df['Filtration flow (m3/h)'].apply(lambda x: x[1]) >= V)]
        if selected_filter.empty:
             print("No suitable vessel found for the given flow rate.")
             return None, None, None
        selected_filter = selected_filter.iloc[0]
        Fl = selected_filter['Filtration flow (m3/h)']
        H = selected_filter["Height (m)"]
        D = selected_filter["Diameter (mm)"]
        print(f"The pressurized vessel diameter is {D:.3f}mm")
        print(f"The height of each pressurized vessel is {H:.3f} m")
       # Calculate filter length and width (assuming rectangular area filter)
        D_m=D/1000
        filterAreaperunit = mh.ceil(1.1*(3.14*D_m**2/4))
        filterArea=mh.ceil(filterAreaperunit*units*lanes)
        print(f"The pressurized vessel unit area is {filterAreaperunit:}m2")
        print(f"The required area for the {units*lanes} pressurized vessels for the rapid sand filtration area is {filterArea}m2")
        Ht=1.1*H+30+10 # 30m is the pressure head and 10m are the friction losses and H is the height of the vessel
        rapidsandhead=Ht
        print(f"The hydraulic losses during the backwash is {rapidsandhead} m")
       # Power and energy calculations
        Phydr=(1000* 9.81*V*Ht/3600)/1000 #Power of the pump
        Pact=Phydr/0.7 # 70% efficiency
        Ptot=Pact*units*lanes # Total power consumption
        Pact=mh.ceil(Pact)
        Ptot=mh.ceil(Ptot)
        print(f"The power consumption required per unit is {Pact} kW:")
        print(f"The total power consumption of the rapid sand filtration area is {Ptot}kW")
        Etot=mh.ceil(Ptot*24)
        print(f"The daily energy consumption is {Etot}kWh")
        print('Assumption: Backwash hydraulic loading rate equal is 50m/h, backwash duration is 10min. Always one unit backwash once per week')
        Qbackwash=filterAreaperunit*50 # 50m3/h/m2 backwash flow rate standard
        t=1/6 #10min backwash time
        Vbackwash=Qbackwash * t
        Vbackwash=mh.ceil(1.1*Vbackwash*lanes*units) # 10% increase in backwash volume
        print(f"The minimum backwash volume required for the backwash of all the rapid sand filtration units is {Vbackwash}m3 ")
        Abackwash=Vbackwash/2
        print(f"The required backwash area is {Abackwash}m2")
        Pback=(1000* 9.81*Qbackwash*Ht*1.1/3600)/1000 #Power of the backwash pump
        Pback=mh.ceil(Pback/0.7) # 70% efficiency
        print(f"The power consumption of the backwash pumps is {Pback:}kW")
        Eback=mh.ceil(Ptot*t*lanes*units/7) # Daily 7days projected energy consumption (one day backwash and  days off)
        print(f"The energy consumption of the backwash pumps is {Eback:}kWh")
       # Total area required for the rapid sand filtration process
        rapidfilterprocessArea = mh.ceil(1.2*(filterArea + Abackwash))
        print(f"The total area of the rapid sand filtration process is {rapidfilterprocessArea}m2")
        bed_depth=H
    else:
       while True:
           units_str = input("Number of different units per lane (minimum 1+1 back up):")
           try:
              units = int(units_str)
              if units >= 2:
                   break
              else:
                  print("Invalid input. Please enter a number greater than or equal to 1.")
           except ValueError:
               print("Invalid input. Please enter a valid number.")
       V=flowRate/(lanes*(units-1))
       data = {
         'Height (m)': [3.9, 4.7, 5.20, 5.7, 5.55, 6.05, 6.55, 5.5, 6.0],
         'Diameter (m)': [1.4, 2.0, 2.0, 2.0, 2.5, 2.5, 2.5, 2.8, 2.8],
         'Net filtration area  (m2)': [1.54, 3.14, 3.14, 3.14, 4.9, 4.9, 4.9, 6.15, 6.15],
         'Sand bed height (m)': [1.0, 1.0, 1.5, 2.0, 1.0, 1.5, 2.0, 1.0, 1.5],
         'Filtration flow (m3/h)': [(0.001, 6), (6, 14.8), (14.8, 16.4),(16.4, 17.9), (17.9, 27.3),(27.3, 29.7), (29.7, 32.2),(32.2, 33.9), (33.9, 37)],
         'Installed Power (kW)': [3.0, 5.5, 5.5, 5.5, 9.2, 9.2, 9.2, 11, 11]}

       # Create DataFrame
       df = pd.DataFrame(data)
       # Print the DataFrame for debugging
       print("Moving Bed Filtration Vessels Technical parameters:")
       print(df)

        # Find the appropriate line based on the given flow rate
       selected_filter = df[(df['Filtration flow (m3/h)'].apply(lambda x: x[0]) < V) &
                              (df['Filtration flow (m3/h)'].apply(lambda x: x[1]) >= V)]
       if selected_filter.empty:
            print("No suitable moving bed filtration found for the given flow rate.")
            return None, None, None
       selected_filter = selected_filter.iloc[0]
       Fl = selected_filter['Filtration flow (m3/h)']
       H = selected_filter["Height (m)"]
       D = selected_filter["Diameter (m)"]
       A = selected_filter["Net filtration area  (m2)"]
       Hs = selected_filter["Sand bed height (m)"]
       Punit = selected_filter["Installed Power (kW)"]
       print(f"The moving bed filtration unit diameter is {D:.3f}m")
       print(f"The height of the moving bed filtration unit is {H:.3f} m")
       filterAreaperunit = mh.ceil(1.1*(3.14*D**2/4))
       filterArea=mh.ceil(filterAreaperunit*units*lanes)
       print(f"The pressurized vessel unit area is {filterAreaperunit}m2")
       print(f"The required area for the {units*lanes} pressurized vessels for the rapid sand filtration area is {filterArea}m2")
       # Total power and energy consumption
       Ptot=Punit*units*lanes
       ptot=mh.ceil(Ptot)
       print(f"The total power consumption for the rapid sand filtration area is {Ptot}kW")
       Etot=Ptot*24
       Etot=mh.ceil(Etot)
       print(f"The daily energy consumption is {Etot}kWh")
       Eback=0
       # Total head losses
       grain_size_m1=0.6/1000
       grain_size_m2=0.3/1000
       vel_unit=(V/A)/3600 # superficial velocity in m/s
       term1=waterkinvisc/g
       term22=(1 - anthrpor)**2 / anthrpor**3
       term32=vel_unit/grain_size_m2**2
       rapidsandhead2=180*term1*term22*term32*Hs
       term21=(1 - sandpor)**2 / sandpor**3
       term31=vel_unit/grain_size_m1**2
       rapidsandhead1=180*term1*term21*term31*Hs
       rapidsandhead=mh.ceil(rapidsandhead1+rapidsandhead2)
       print(f"The hydraulic losses in the rapid sand filtration units is {rapidsandhead} m")
       # Total area required for the rapid sand filtration process
       rapidfilterprocessArea = mh.ceil(1.2*(filterArea))
       print(f"The total area of the rapid sand filtration process is {rapidfilterprocessArea} m2")
       bed_depth=H
    return rapidfilterprocessArea,Etot,Eback,rapidsandhead,bed_depth
